# relationships eda (weighted)

_Jordan_

In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns 
import matplotlib.pyplot as plt
import statsmodels.api as sm

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



In [ ]:
# Define the technical columns we need
cols = ['RECENT_CHECKUP', 'HAS_EXERCISE', 'GOOD_HEALTH', 'HAS_PERSONAL_DOCTOR', 
        'PHYSHLTH', 'MENTHLTH', '_BMI5_SCALED', '_LLCPWT_POOLED', 
        'EVER_SMOKED', 'HAS_DEPRESSION']

# Load the data
df = pd.read_parquet('/kaggle/input/datasets/ajenks/brfss-2020-2024-cleaned-and-weighted/brfss_2020_2024_pooled_eda.parquet', columns=cols)
df = df.dropna()

# Rename columns to actual descriptors
df = df.rename(columns={
    'GOOD_HEALTH': 'Good General Health',
    'HAS_EXERCISE': 'Physical Activity',
    '_BMI5_SCALED': 'BMI Scaled',
    'RECENT_CHECKUP': 'Recent Checkup',
    'EVER_SMOKED': 'Smoking History',
    'PHYSHLTH': 'Physical Unhealthy Days',
    'MENTHLTH': 'Mental Unhealthy Days',
    'HAS_PERSONAL_DOCTOR': 'Has Personal Doctor',
    'HAS_DEPRESSION': 'Depression Diagnosis',
    '_LLCPWT_POOLED': 'Survey Weight'
})

# Verify weighting with new names
unweighted_rate = df['Physical Activity'].mean() * 100
weighted_rate = np.average(df['Physical Activity'], weights=df['Survey Weight']) * 100

print(f'Unweighted Physical Activity Rate: {unweighted_rate:.1f}%')
print(f'Weighted Physical Activity Rate: {weighted_rate:.1f}%')

In [ ]:
plt.figure(figsize=(10, 6))

violin_plot_df = df.copy()
violin_plot_df['Physical_Activity_Label'] = violin_plot_df['Physical Activity'].map({1.0: 'Active', 0.0: 'Inactive'})

sns.set_style("whitegrid")
ax = sns.violinplot(x="Physical_Activity_Label", y="Physical Unhealthy Days", data=violin_plot_df, palette="muted")

ax.set_xlabel("Physical Activity Status", fontsize=12)
ax.set_ylabel("Number of Physically Unhealthy Days", fontsize=12)
plt.title("Distribution of Physical Unwellness by Activity Status", fontsize=14)

plt.show()

active vs inactive — active group sits near 0 unhealthy days, inactive group has a long tail to 30. holds up under the survey weights.

In [ ]:
def weighted_corr(df, cols, weight_col):
    def weighted_cov(x, y, w):
        return np.sum(w * (x - np.average(x, weights=w)) * (y - np.average(y, weights=w))) / np.sum(w)
    def weighted_correlation(x, y, w):
        return weighted_cov(x, y, w) / np.sqrt(weighted_cov(x, x, w) * weighted_cov(y, y, w))

    n = len(cols)
    corr_matrix = pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)
    for i in range(n):
        for j in range(i + 1, n):
            c = weighted_correlation(df[cols[i]], df[cols[j]], df[weight_col])
            corr_matrix.iloc[i, j] = corr_matrix.iloc[j, i] = c
    return corr_matrix

# Select variables using new descriptors
analysis_cols = ['Physical Unhealthy Days', 'Mental Unhealthy Days', 'BMI Scaled', 'Has Personal Doctor', 'Recent Checkup']
weighted_matrix = weighted_corr(df, analysis_cols, 'Survey Weight')

plt.figure(figsize=(10, 8))
sns.heatmap(weighted_matrix, annot=True, cmap='RdBu_r', center=0)
plt.title("Weighted Correlation: Health Outcomes vs. Healthcare Access")
plt.show()

weighted corr matrix. phys + mental unhealthy days move together. personal doctor is the strongest correlate of having a recent checkup. bmi shows a small positive corr with phys unhealthy days.

In [ ]:
weighted_plot_df = df.groupby('Physical Activity').apply(
    lambda x: np.average(x['Good General Health'], weights=x['Survey Weight']) * 100
).reset_index(name='Good Health Percentage')

# Map binary values to text for the plot
weighted_plot_df['Physical Activity'] = weighted_plot_df['Physical Activity'].replace({1.0: "Active", 0.0: "Inactive"})

plt.figure(figsize=(10, 6))
ax = sns.barplot(x="Physical Activity", y="Good Health Percentage", data=weighted_plot_df, palette="viridis")

plt.title("Weighted Impact of Physical Activity on General Health", fontsize=14)
plt.ylabel("Percentage Reporting 'Good General Health' (%)")
plt.xlabel("Physical Activity Status")

for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', xytext=(0, 9), textcoords='offset points')

plt.show()

switching to weighted percentages: active adults are about 2x as likely to report good general health as inactive.

In [ ]:
# Create a weighted pivot table with descriptive names
pivot_weighted = df.groupby(['Depression Diagnosis', 'Physical Activity']).apply(
    lambda x: np.average(x['Good General Health'], weights=x['Survey Weight']) * 100
).reset_index(name='Good Health Rate')

# Map for plot readability
pivot_weighted['Depression Diagnosis'] = pivot_weighted['Depression Diagnosis'].replace({1.0: "Diagnosed", 0.0: "Not Diagnosed"})
pivot_weighted['Physical Activity'] = pivot_weighted['Physical Activity'].replace({1.0: "Physically Active", 0.0: "Inactive"})

g = sns.catplot(
    data=pivot_weighted, x="Depression Diagnosis", y="Good Health Rate", col="Physical Activity",
    kind="bar", palette="magma", height=5, aspect=0.8
)

g.fig.suptitle("Weighted Health Status: Depression vs. Physical Activity", fontsize=16, y=1.05)
g.set_axis_labels("Depression Diagnosis Status", "Weighted % in Good General Health")
plt.show()

interaction between activity and a depression dx. activity buffers the gap a lot — active+depressed often report better health than inactive+no-depression.

In [ ]:
# Define features using descriptors
target = 'Good General Health'
features = ['Physical Activity', 'BMI Scaled', 'Recent Checkup', 'Smoking History']

y = df[target]
X = df[features]
weights = df['Survey Weight']

# Add constant for intercept
X = sm.add_constant(X)

# Fit the Weighted GLM
weighted_model = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=weights).fit()

# The summary table will now use the descriptive column names
print(weighted_model.summary())

# Weighted Odds Ratios
odds_ratios = np.exp(weighted_model.params)
print("\n--- Weighted Odds Ratios ---")
print(odds_ratios)

weighted glm. active = ~3x odds of good health. bmi and ever-smoked drag odds down. recent checkup is *negative* — likely because the people seeing doctors most are the ones with chronic stuff.